In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

/Users/alesy/kursach/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pyarrow.parquet as pq
df = pd.read_parquet(
    '/Users/alesy/kursach/data/interactions_audio_v2.1.parquet',
    columns=['user_id', 'song_id', 'play_count']
).head(600000)
df.shape

(600000, 3)

In [3]:
df['user_id'].nunique()

21781

In [ ]:
user_cat = df["user_id"].astype("category")
song_cat = df["song_id"].astype("category")
alpha = 40

# перед построением CSR: 20% взаимодействий в test
np.random.seed(42)
mask = np.random.rand(len(df)) < 0.8
df_train, df_test = df[mask], df[~mask]

# CSR только на train
user_idx_tr = user_cat.cat.codes.values[mask]
song_idx_tr = song_cat.cat.codes.values[mask]
conf_tr = 1 + alpha * df_train['play_count'].values

train_mat = csr_matrix(
    (conf_tr.astype(np.float32), (user_idx_tr, song_idx_tr)),
    shape=(user_cat.cat.categories.size, song_cat.cat.categories.size)
)

model = AlternatingLeastSquares(factors=8, regularization=0.1, iterations=15)
model.fit(train_mat)


100%|██████████| 15/15 [00:02<00:00,  7.47it/s]


In [5]:
import math

# для каждого test-юзера — какие песни в test (ground truth)
test_by_user = df_test.groupby('user_id')['song_id'].apply(set).to_dict()

K = 10
precisions, recalls, ndcgs = [], [], []

user_to_int = {u: i for i, u in enumerate(user_cat.cat.categories)}
song_categories = song_cat.cat.categories   # тоже один раз

for u_str, truth_songs in test_by_user.items():
    u_int = user_to_int.get(u_str)
    if u_int is None or not truth_songs: continue

    ids, _ = model.recommend(u_int, train_mat[u_int], N=K)
    rec_songs = set(song_cat.cat.categories[ids])

    hits = rec_songs & truth_songs
    precisions.append(len(hits) / K)
    recalls.append(len(hits) / len(truth_songs))

    # NDCG
    ranked = [song_cat.cat.categories[i] for i in ids]
    dcg = sum(1/math.log2(i+2) for i,s in enumerate(ranked) if s in truth_songs)
    idcg = sum(1/math.log2(i+2) for i in range(min(len(truth_songs), K)))
    ndcgs.append(dcg/idcg if idcg>0 else 0)

print(f'Precision@{K}: {np.mean(precisions):.4f}')
print(f'Recall@{K}:    {np.mean(recalls):.4f}')
print(f'NDCG@{K}:      {np.mean(ndcgs):.4f}')


Precision@10: 0.0517
Recall@10:    0.1122
NDCG@10:      0.0962


In [6]:
import sys, os
sys.path.insert(0, os.path.expanduser('~/kursach'))
from distance_metrics import evaluate_all, to_dataframe
from sklearn.preprocessing import normalize
import pandas as pd

# 1. эмбеддинги
ufac = model.user_factors    # (n_users, 50)
ifac = model.item_factors    # (n_items, 50)

# 2. id в порядке строк ufac/ifac
user_ids_order = user_cat.cat.categories.to_numpy()
song_ids_order = song_cat.cat.categories.to_numpy()

# 3. что юзер видел в train (исключим из рекомендаций)
train_seen = df_train.groupby('user_id')['song_id'].apply(set).to_dict()

# 4а. RAW: сырые эмбеддинги — длины несут сигнал
results_raw = evaluate_all(
    P=ufac, M=ifac,
    user_ids=user_ids_order, item_ids=song_ids_order,
    train_seen=train_seen, test_truth=test_by_user, K=10,
)

# 4б. NORMALIZED: L2-нормировка строк, все метрики работают на единичной сфере
ufac_n = normalize(ufac, axis=1)
ifac_n = normalize(ifac, axis=1)
results_norm = evaluate_all(
    P=ufac_n, M=ifac_n,
    user_ids=user_ids_order, item_ids=song_ids_order,
    train_seen=train_seen, test_truth=test_by_user, K=10,
)

# объединяем в одну таблицу
table = pd.concat({
    'raw':  to_dataframe(results_raw),
    'norm': to_dataframe(results_norm),
}, names=['mode', 'metric'])
print(table)
table.to_csv('results_als.csv')


[17:04:05] [1/5] metric 'dot': computing scores...
[17:04:05]   scores computed in 0.2s, shape=(21781, 14303)
[17:04:05]   copy scores (21781x14303) -> float64
[17:04:07]   masking train interactions...
[17:04:13]   masking done in 6.6s, computing top-K + metrics...
[17:04:14]     5000/21781 users  rate=6102/s  eta=3s
[17:04:15]     10000/21781 users  rate=6942/s  eta=2s
[17:04:15]     15000/21781 users  rate=7372/s  eta=1s
[17:04:16]     20000/21781 users  rate=7647/s  eta=0s
[17:04:16]   done: 19882 users evaluated in 2.8s
[17:04:16]   'dot' result: P@10=0.0517 R@10=0.1122 NDCG@10=0.0962
[17:04:16] [2/5] metric 'cos': computing scores...
[17:04:16]   scores computed in 0.2s, shape=(21781, 14303)
[17:04:16]   copy scores (21781x14303) -> float64
[17:04:18]   masking train interactions...
[17:04:28]   masking done in 10.7s, computing top-K + metrics...
[17:04:29]     5000/21781 users  rate=6793/s  eta=2s
[17:04:30]     10000/21781 users  rate=7203/s  eta=2s
[17:04:30]     15000/21781 u

# Без валидации

In [7]:
user_cat = df["user_id"].astype("category")
song_cat = df["song_id"].astype("category")

user_idx = user_cat.cat.codes.values
song_idx = song_cat.cat.codes.values

# уверенность
alpha = 40
confidence = 1 + alpha * df["play_count"].values

# собираем CSR: строки = пользователи, столбцы = песни, значения = confidence
user_items = csr_matrix(
    (confidence.astype(np.float32), (user_idx, song_idx)),
    shape=(user_cat.cat.categories.size, song_cat.cat.categories.size)
)
user_items

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 600000 stored elements and shape (21781, 14303)>

In [8]:
model = AlternatingLeastSquares(factors=50, regularization=0.1, iterations=15)
model.fit(user_items)

100%|██████████| 15/15 [00:02<00:00,  6.38it/s]


In [9]:
# возьмём случайного юзера
uid_internal = 50  # внутренний индекс (0..4062)
user_id_str = user_cat.cat.categories[uid_internal]
print(f'user: {user_id_str}')

# что он уже слушал
seen_mask = user_items[uid_internal].toarray().ravel() > 0
seen_songs = song_cat.cat.categories[seen_mask]
seen_names = df[df.song_id.isin(seen_songs)][['name','artists']].drop_duplicates().head(10)
print('СЛУШАЛ:')
print(seen_names)

# рекомендации
ids, scores = model.recommend(uid_internal, user_items[uid_internal], N=10)
rec_songs = song_cat.cat.categories[ids]
recs = df[df.song_id.isin(rec_songs)][['name','artists','genre_root']].drop_duplicates()
print('\nРЕКОМЕНДУЕТ:')
print(recs)


user: 00ab8753e09a9a0badbeb44a2433f28a4d305812


KeyError: "None of [Index(['name', 'artists'], dtype='str')] are in the [columns]"